# WideBind — Cloud Training (Сферум / любой GPU)
D=4096, L=32, G=32, k=32, ~293M params.

**Перед запуском:** загрузите в среду:
- `core/` (config.py, model.py, lambda_utils.py)
- `token_stream_DETECT_clean.bin`
- `token_stream_ACTION_clean.bin`

Либо `git clone` репозитория.

**Ключевые фичи включенные по умолчанию:**
- λ-d иерархия (d=3) — все константы из λ₃≈1.839
- PH+B readout — per-token bias для частотного приоритета
- tie_bind / tie_mirror_proj — автоэнкодерное ограничение, −557K params
- alpha=0.98 init — |1-alpha|=0.016 (+60% temporal gradient)
- AdaptiveController per-layer

Логирование: `|1-alpha|` — отклонение alpha от 1.0 (должно >0.01 при обучении).

In [ ]:
# --- Setup paths ---
import sys, os

notebook_dir = os.path.abspath('')
if os.path.basename(notebook_dir) == 'notebooks':
    project_root = os.path.dirname(notebook_dir)
else:
    project_root = notebook_dir

sys.path.insert(0, project_root)
DATA_DIR = project_root

print(f'Project root: {project_root}')
print(f'Data dir: {DATA_DIR}')

In [ ]:
# --- Импорт и конфиг ---
from core import WideBindConfig, WideBindStack, MirrorLRScheduler

# Все новые фичи включены по умолчанию:
# tie_bind=True, tie_mirror_proj=True, lambda_d_enabled=True,
# zeckendorf_readout=False, temporal_zeckendorf=False
cfg = WideBindConfig(
    D=4096, n_layers=32, bind_K=64,
    mlp_groups=32, mlp_expand=4,
    mirror_k=32, seq_len=128,
    lr=3e-4, max_steps=500000,
    warmup_steps=2000,
    grad_clip=1.0, conv_kernel=48,
    scheduler='mirror',
    batch_size=2,
    data_dir=DATA_DIR,
    save_dir=os.path.join(project_root, 'checkpoints'),
    log_dir=os.path.join(project_root, 'logs'),
)

model = WideBindStack(cfg)
print(f'Model: {model.param_count():,} params ({model.param_count()/1e6:.2f}M)')

# Проверки
a = model.layers[0].mirror.alpha
print(f'Mirror alpha shape: {list(a.shape)} (expected [32])')
assert a.shape == (32,)
print(f'tie_bind: {cfg.tie_bind} |1-alpha| init={abs(1-a.mean().item()):.4f}')
print('OK')

In [ ]:
# --- Загрузка данных ---
import glob, numpy as np

stream_files = sorted(glob.glob(os.path.join(DATA_DIR, 'token_stream_*_clean.bin')))
if not stream_files:
    stream_files = sorted(glob.glob(os.path.join(DATA_DIR, 'token_stream_*.bin')))
if not stream_files:
    raise FileNotFoundError(f'No token_stream_*.bin files in {DATA_DIR}')

streams = [np.fromfile(f, dtype=np.uint16) for f in stream_files]
total_tokens = sum(len(s) for s in streams)
print(f'Found {len(stream_files)} files:')
for f in stream_files:
    sz = os.path.getsize(f) / 1e9
    print(f'  {os.path.basename(f)}: {sz:.2f} GB')
print(f'Total: {total_tokens:,} tokens')

In [ ]:
# --- Device и batch size (auto-fit, начиная с 2) ---
import torch, gc

device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_name = torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'
vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if device == 'cuda' else 0
print(f'Device: {device} ({gpu_name})')
print(f'VRAM: {vram:.1f} GB')

model = model.to(device)

for bs in [2, 1]:
    torch.cuda.empty_cache(); gc.collect()
    try:
        x = torch.randint(0, cfg.vocab, (bs, cfg.seq_len), device=device)
        h = model.embed_tokens(x)
        out, _, _ = model(h, None)
        out[:, :1].sum().backward(); model.zero_grad()
        del x, h, out; torch.cuda.empty_cache()
        cfg.batch_size = bs
        print(f'Batch size: {bs} — OK')
        break
    except RuntimeError:
        model.zero_grad(); torch.cuda.empty_cache(); gc.collect()
        print(f'Batch size {bs}: OOM')
else:
    cfg.batch_size = 1
    print('Batch size: 1 (minimum!)')

In [ ]:
# --- Оптимизатор и MirrorLRScheduler ---
optimizer = torch.optim.AdamW(model.param_groups(), betas=(0.9, 0.95))
scheduler = MirrorLRScheduler(
    model, optimizer, cfg.lr,
    warmup=cfg.warmup_steps,
    target_var=cfg.target_var,
    mag_threshold=cfg.mag_threshold,
    lr_min_ratio=cfg.lr_min_ratio,
    max_decay_steps=cfg.max_decay_steps,
    var_min_for_lr_decay=cfg.var_min_for_lr_decay,
)

os.makedirs(cfg.save_dir, exist_ok=True)
print(f'Optimizer: AdamW (betas 0.9, 0.95)')
print(f'Scheduler: Mirror (target_var={cfg.target_var})')

In [ ]:
# --- Training loop ---
import time, math

@torch.no_grad()
def evaluate():
    model.eval()
    total, cnt = 0.0, 0
    for s in streams:
        off = max(len(s) // 4, cfg.batch_size * cfg.seq_len + 1)
        for _ in range(200):
            if off + cfg.batch_size * cfg.seq_len + 1 > len(s):
                off = 0; break
            ch = s[off:off+cfg.batch_size*cfg.seq_len+1]
            x = torch.from_numpy(ch[:cfg.batch_size*cfg.seq_len].copy()).long().view(cfg.batch_size, cfg.seq_len).to(device)
            y = torch.from_numpy(ch[1:cfg.batch_size*cfg.seq_len+1].copy()).long().view(cfg.batch_size, cfg.seq_len).to(device)
            off += cfg.batch_size * cfg.seq_len
            h = model.embed_tokens(x); out, _, _ = model(h, None)
            total += model.compute_loss(out, y).item(); cnt += 1
    model.train()
    return total / max(cnt, 1)

def get_batch(s, off):
    if off + cfg.batch_size * cfg.seq_len + 1 > len(s):
        return None, None, 0
    ch = s[off:off+cfg.batch_size*cfg.seq_len+1]
    x = torch.from_numpy(ch[:cfg.batch_size*cfg.seq_len].copy()).long().view(cfg.batch_size, cfg.seq_len).to(device)
    y = torch.from_numpy(ch[1:cfg.batch_size*cfg.seq_len+1].copy()).long().view(cfg.batch_size, cfg.seq_len).to(device)
    return x, y, off + cfg.batch_size * cfg.seq_len

# --- Resume ---
start_step = 0; best_val = float('inf')
import re
ckpt_files = sorted(glob.glob(os.path.join(cfg.save_dir, '*.pt')))
if ckpt_files:
    def step_key(p):
        m = re.search(r'step_(\d+)\.pt', os.path.basename(p))
        return int(m.group(1)) if m else 0
    ckpt_files.sort(key=step_key)
    latest = ckpt_files[-1]
    print(f'Resuming from {latest}')
    ckpt = torch.load(latest, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'], strict=False)
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_step = ckpt['step']
    best_val = ckpt.get('best_val_loss', float('inf'))
    print(f'  Resume at step {start_step}')

si, off, state = 0, 0, None
tokens_seen = 0; t0 = time.time()

print(f'\n{"="*60}')
print(f'Training: step {start_step} -> {cfg.max_steps}'
      f'  bs={cfg.batch_size}  seq={cfg.seq_len}')
print(f'{"="*60}')
print(f'  {"step":>6}  {"loss":>8}  {"|1-alpha|":>10}  {"gate_var":>8}  {"tok/s":>6}')
print(f'  {"-"*42}')

try:
    for step in range(start_step, cfg.max_steps):
        model.train()
        s = streams[si]
        x, y, off = get_batch(s, off)
        if x is None:
            si = (si + 1) % len(streams)
            s = streams[si]
            _, _, off = get_batch(s, 0)
            state = None
            x, y, off = get_batch(s, off)

        h = model.embed_tokens(x)
        out, state, _ = model(h, state)
        loss = model.compute_loss(out, y)

        loss.backward()
        if cfg.grad_clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()
        tokens_seen += cfg.batch_size * cfg.seq_len

        if step % cfg.log_interval == 0:
            with torch.no_grad():
                idiff = torch.stack([
                    (1.0 - l.mirror.alpha.data).abs().mean()
                    for l in model.layers
                ]).mean().item()
                gv = torch.stack([
                    l.mirror._last_gates.var()
                    for l in model.layers
                ]).mean().item()
            dt = time.time() - t0
            print(f'  {step:>6}  {loss.item():>8.4f}  {idiff:>10.6f}  {gv:>8.6f}  {tokens_seen/max(dt,1e-8):>6.0f}')

        if step > 0 and step % cfg.eval_interval == 0:
            vl = evaluate()
            print(f'  EVAL step={step}: loss={vl:.4f} ppl={math.exp(vl):.1f}')
            if vl < best_val:
                best_val = vl
                torch.save({
                    'step': step, 'model': model.state_dict(),
                    'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(),
                    'best_val_loss': best_val, 'cfg': cfg,
                }, os.path.join(cfg.save_dir, 'best.pt'))
                print(f'  -> New best!')

        if step > 0 and step % cfg.save_interval == 0:
            torch.save({
                'step': step, 'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'best_val_loss': best_val, 'cfg': cfg,
            }, os.path.join(cfg.save_dir, f'step_{step}.pt'))
            print(f'  Checkpoint saved (step {step})')

except KeyboardInterrupt:
    torch.save({
        'step': step, 'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'best_val_loss': best_val, 'cfg': cfg,
    }, os.path.join(cfg.save_dir, f'interrupt_step_{step}.pt'))
    print(f'\nInterrupted. Saved.')

In [ ]:
# --- Анализ alpha per layer ---
with torch.no_grad():
    print('Alpha per layer:')
    for i, l in enumerate(model.layers):
        a = l.mirror.alpha.data
        ad = (1.0 - a).abs().mean().item()
        gv = l.mirror._last_gates.var().item() if hasattr(l.mirror, '_last_gates') else 0
        print(f'  L{i:>2}: alpha={a.mean().item():.4f} |1-alpha|={ad:.6f} gate_var={gv:.6f}')